In [ ]:
# Installs if needed
!pip install torch torchvision torchaudio
!pip install transformers
!pip install datasets
!pip install tqdm
!pip install accelerate

In [ ]:
# Needed imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from transformers import RobertaModel
from transformers import RobertaConfig
from transformers import RobertaTokenizer
from transformers import get_cosine_schedule_with_warmup
from datasets import load_dataset
from tqdm import tqdm
import random
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set the seet
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# Roberta large with LoRA adapters
class LoRALayer:
    def __init__(self, r: int, lora_alpha: int, lora_dropout: float):
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r if r > 0 else 0
        self.lora_dropout = nn.Dropout(p=lora_dropout)
        self.merged = False

class LoRALinear(nn.Module, LoRALayer):
    def __init__(self, in_features: int, out_features: int, r: int = 0,
                 lora_alpha: int = 1, lora_dropout: float = 0.0):
        nn.Module.__init__(self)
        LoRALayer.__init__(self, r, lora_alpha, lora_dropout)
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.zeros(out_features, in_features))
        self.weight.requires_grad = False
        self.bias = None
        if r > 0:
            self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
            self.lora_B = nn.Parameter(torch.zeros(out_features, r))
            self.scaling = self.lora_alpha / self.r
        else:
            self.lora_A = None
            self.lora_B = None
        self.reset_parameters()

    def reset_parameters(self):
        if hasattr(self, 'lora_A'):
            nn.init.normal_(self.lora_A, std=0.01)
            nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        result = F.linear(x, self.weight, self.bias)
        if self.r > 0 and not self.merged:
            x = self.lora_dropout(x)
            result = result + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return result

class LoRAConfig:
    def __init__(self, r=8, lora_alpha=16, lora_dropout=0.0,
                 target_modules=None):
        self.r = r
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        if target_modules is None:
            target_modules = ["attention.self.query", "attention.self.key"]
        self.target_modules = target_modules

def apply_lora(model, config: LoRAConfig):
    replaced = 0
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        for target in config.target_modules:
            if target not in name:
                continue
            in_f, out_f = module.in_features, module.out_features
            w = module.weight.data.clone()
            b = module.bias.data.clone() if module.bias is not None else None
            new = LoRALinear(in_f, out_f, r=config.r,
                             lora_alpha=config.lora_alpha,
                             lora_dropout=config.lora_dropout)
            new.weight.data = w
            if b is not None:
                new.bias = nn.Parameter(b)
            parent_name, child_name = name.rsplit('.', 1)
            setattr(model.get_submodule(parent_name), child_name, new)
            print(f"  applied LoRA to {name}: {in_f * out_f:,} -> {2 * config.r * in_f:,} params")
            replaced += 1
            break
    print(f"total LoRA layers: {replaced}")
    return model

# LoRA with an extra square C matrix in between A and B
class LoRALinear3(nn.Module, LoRALayer):
    def __init__(self, in_features: int, out_features: int, r: int = 0,
                 lora_alpha: int = 1, lora_dropout: float = 0.0):
        nn.Module.__init__(self)
        LoRALayer.__init__(self, r, lora_alpha, lora_dropout)
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.zeros(out_features, in_features))
        self.weight.requires_grad = False
        self.bias = None
        if r > 0:
            self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
            self.lora_C = nn.Parameter(torch.eye(r))
            self.lora_B = nn.Parameter(torch.zeros(out_features, r))
            self.scaling = lora_alpha / r
        else:
            self.lora_A = self.lora_C = self.lora_B = None
        self.reset_parameters()

    def reset_parameters(self):
        if hasattr(self, 'lora_A'):
            nn.init.normal_(self.lora_A, std=0.01)
            nn.init.zeros_(self.lora_B)

    def forward(self, x):
        result = F.linear(x, self.weight, self.bias)
        if self.r > 0 and not self.merged:
            x = self.lora_dropout(x)
            result = result + (x @ self.lora_A.T @ self.lora_C @ self.lora_B.T) * self.scaling
        return result

class LoRAConfig3:
    def __init__(self, r=8, lora_alpha=16, lora_dropout=0.0,
                 target_modules=None):
        self.r = r
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        if target_modules is None:
            target_modules = ["attention.self.query", "attention.self.key"]
        self.target_modules = target_modules

def apply_lora3(model, config: LoRAConfig3):
    replaced = 0
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        for target in config.target_modules:
            if target not in name:
                continue
            in_f, out_f = module.in_features, module.out_features
            w = module.weight.data.clone()
            b = module.bias.data.clone() if module.bias is not None else None
            new = LoRALinear3(in_f, out_f, r=config.r,
                              lora_alpha=config.lora_alpha,
                              lora_dropout=config.lora_dropout)
            new.weight.data = w
            if b is not None:
                new.bias = nn.Parameter(b)
            parent_name, child_name = name.rsplit('.', 1)
            setattr(model.get_submodule(parent_name), child_name, new)
            p = config.r * in_f + config.r * config.r + out_f * config.r
            print(f"  applied LoRA3 to {name}: {in_f * out_f:,} -> {p:,} params")
            replaced += 1
            break
    print(f"total LoRA3 layers: {replaced}")
    return model

In [ ]:
# GLUE dataset functions
class GLUEDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_len=128):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        item = self.dataset[idx]
        enc = self.tokenizer(item['sentence'], max_length=self.max_len,
                             padding='max_length', truncation=True,
                             return_tensors='pt')
        return enc['input_ids'].squeeze(), enc['attention_mask'].squeeze(), item.get('label', 0)

class RobertaEncoder(nn.Module):
    def __init__(self, model_name="roberta-large", freeze_encoder=True,
                 lora_config=None):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        self.hidden_size = self.roberta.config.hidden_size
        if lora_config is not None:
            apply_lora(self.roberta, lora_config)
        if freeze_encoder:
            for name, p in self.roberta.named_parameters():
                if not any(k in name for k in ['lora_A', 'lora_B']):
                    p.requires_grad = False
    def forward(self, input_ids, attention_mask):
        return self.roberta(input_ids=input_ids, attention_mask=attention_mask).pooler_output

class RobertaClassifier(nn.Module):
    def __init__(self, model_name="roberta-large", num_classes=2, dropout=0.1,
                 lora_config=None):
        super().__init__()
        self.encoder = RobertaEncoder(model_name, freeze_encoder=True,
                                       lora_config=lora_config)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.encoder.hidden_size, num_classes))
    def forward(self, input_ids, attention_mask):
        return self.classifier(self.encoder(input_ids, attention_mask))

class RobertaEncoder3(nn.Module):
    def __init__(self, model_name="roberta-large", freeze_encoder=True,
                 lora_config=None):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        self.hidden_size = self.roberta.config.hidden_size
        if lora_config is not None:
            apply_lora3(self.roberta, lora_config)
        if freeze_encoder:
            for name, p in self.roberta.named_parameters():
                if not any(k in name for k in ['lora_A', 'lora_B', 'lora_C']):
                    p.requires_grad = False
    def forward(self, input_ids, attention_mask):
        return self.roberta(input_ids=input_ids, attention_mask=attention_mask).pooler_output

class RobertaClassifier3(nn.Module):
    def __init__(self, model_name="roberta-large", num_classes=2, dropout=0.1,
                 lora_config=None):
        super().__init__()
        self.encoder = RobertaEncoder3(model_name, freeze_encoder=True,
                                        lora_config=lora_config)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.encoder.hidden_size, num_classes))
    def forward(self, input_ids, attention_mask):
        return self.classifier(self.encoder(input_ids, attention_mask))

def train_epoch(model, loader, opt, sched, criterion):
    model.train()
    total = 0
    for batch in tqdm(loader):
        ids, mask, labels = [x.to(device) for x in batch]
        opt.zero_grad()
        loss = criterion(model(ids, mask), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        total += loss.item()
    return total / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    loss_total = correct = total = 0
    with torch.no_grad():
        for batch in loader:
            ids, mask, labels = [x.to(device) for x in batch]
            logits = model(ids, mask)
            loss_total += criterion(logits, labels).item()
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += labels.size(0)
    return loss_total / len(loader), correct / total

In [ ]:
def train_glue(r, data_used, variant='lora'):
    tokenizer = RobertaTokenizer.from_pretrained('roberta-large')
    ds = load_dataset('glue', 'sst2')

    tsize = int(len(ds['train']) * data_used)
    vsize = int(len(ds['validation']) * data_used)
    train_loader = DataLoader(
        GLUEDataset(ds['train'].select(range(tsize)), tokenizer),
        batch_size=32, shuffle=True)
    val_loader = DataLoader(
        GLUEDataset(ds['validation'].select(range(vsize)), tokenizer),
        batch_size=32)

    if variant == 'lora3':
        cfg = LoRAConfig3(r=r, lora_alpha=16, lora_dropout=0.0)
        model = RobertaClassifier3(lora_config=cfg).to(device)
    else:
        cfg = LoRAConfig(r=r, lora_alpha=16, lora_dropout=0.0)
        model = RobertaClassifier(lora_config=cfg).to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p = sum(p.numel() for p in model.parameters())
    print(f"trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")

    opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    steps = len(train_loader) * 6
    sched = get_cosine_schedule_with_warmup(opt,
        num_warmup_steps=int(0.1 * steps), num_training_steps=steps)
    criterion = nn.CrossEntropyLoss()

    best = 0
    for epoch in range(6):
        train_loss = train_epoch(model, train_loader, opt, sched, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        print(f"epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best:
            best = val_acc
            # TODO: save path should be configurable
            torch.save(model.state_dict(), f"best_{variant}_r{r}.pt")
    return best

def plot_table(results, data_used):
    import matplotlib.pyplot as plt
    paper = 0.962

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.axis('off')
    cols = ["Model", "Rank", "Val Acc", "vs Paper"]
    rows = []
    for name, acc in results.items():
        m = "LoRA" if "LoRA3" not in name else "LoRA3"
        r = name.split("_r")[-1]
        rows.append([m, r, f"{acc*100:.2f}%", f"{(acc-paper)*100:+.2f}%"])

    tab = ax.table(cellText=rows, colLabels=cols, cellLoc='center', loc='center')
    tab.auto_set_font_size(False)
    tab.set_fontsize(11)
    tab.scale(1.2, 2.0)
    for j in range(len(cols)):
        tab[0, j].set_facecolor("#2c3e50")
        tab[0, j].set_text_props(color="white", fontweight="bold")
    for i, (name, acc) in enumerate(results.items()):
        c = "#d4edda" if acc == max(results.values()) else ["#f2f2f2", "#ffffff"][i % 2]
        for j in range(len(cols)):
            tab[i+1, j].set_facecolor(c)
    ax.set_title(f"SST-2 results ({data_used*100:.0f}% data, 6 epochs)",
                  fontsize=13, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()

def run_all(data_used=0.5):
    results = {}
    for r in [4, 8]:
        print(f"\n=== LoRA r={r} ===")
        results[f"LoRA_r{r}"] = train_glue(r, data_used, variant='lora')
    for r in [4, 8]:
        print(f"\n=== LoRA3 r={r} ===")
        results[f"LoRA3_r{r}"] = train_glue(r, data_used, variant='lora3')
    plot_table(results, data_used)
    return results

set_seed(42)
run_all(data_used=0.5)